### loading dataset and preprocessing

In [1]:
import re
import emoji
import contractions
from nltk.stem import WordNetLemmatizer
import spacy
from nltk.corpus import stopwords
import nltk
import numpy as np
# Download stopwords if not already
nltk.download("stopwords")
nltk.download("wordnet")

# Load spaCy
nlp = spacy.load("en_core_web_sm")

stop_words = set(stopwords.words("english"))
lemmatizer = WordNetLemmatizer()

def text_preprocessing(text):
    text = emoji.demojize(text)
    text = contractions.fix(text)
    text = text.lower()
    text = re.sub(r'\d{4}-\d{2}-\d{2}', 'DATE', text)
    text = re.sub(r'\d+', "NUM", text)
    text = re.sub(r'\b\w+@\w+\.\w+\b', 'EMAIL', text)
    text = re.sub(r'[^\w\s]', "", text)
    # Tokenize with spaCy
    doc = nlp(text)
    #print([token.text for token in doc])
    # Lemmatize + remove stopwords
    tokens = [
        lemmatizer.lemmatize(token.text) 
        for token in doc 
        if token.text not in stop_words and not token.is_space
    ]

    return  tokens #" ".join(tokens)


[nltk_data] Downloading package stopwords to
[nltk_data]     /home/mudasir/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to /home/mudasir/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


In [52]:
spam_data = {}

with open("/home/mudasir/ankit/NLP/Lab_06/spam.txt" , 'r') as f:
    text = []
    label = []
    for line in f:
        label.append(int(line.split()[-1]))
        text.append(" ".join(line.split()[:-1]))
    spam_data['text'] = text
    spam_data['label'] = label

In [53]:
print(spam_data['text'][:5])
print(spam_data['label'][:5])

['Go until jurong point, crazy.. Available only in bugis n great world la e buffet... Cine there got amore wat...', 'Ok lar... Joking wif u oni...', 'U dun say so early hor... U c already then say...', "Nah I don't think he goes to usf, he lives around here though", 'Even my brother is not like to speak with me. They treat me like aids patent.']
[0, 0, 0, 0, 0]


In [54]:
preprocessed_spam_data = {}
preprocessed_spam_data['text'] = [text_preprocessing(t) for t in spam_data['text']]
preprocessed_spam_data['label'] = spam_data['label']


In [55]:
print(preprocessed_spam_data['text'][:5])
print(preprocessed_spam_data['label'][:5]) 


[['go', 'jurong', 'point', 'crazy', 'available', 'bugis', 'n', 'great', 'world', 'la', 'e', 'buffet', 'cine', 'got', 'amore', 'wat'], ['ok', 'lar', 'joking', 'wif', 'oni'], ['dun', 'say', 'early', 'hor', 'c', 'already', 'say'], ['nah', 'think', 'go', 'usf', 'life', 'around', 'though'], ['even', 'brother', 'like', 'speak', 'treat', 'like', 'aid', 'patent']]
[0, 0, 0, 0, 0]


In [56]:
from torch.utils.data import Dataset, random_split

class SpamDataset(Dataset):
    def __init__(self, texts, labels):
        self.texts = texts
        self.labels = labels
    
    def __len__(self):
        return len(self.texts)
    
    def __getitem__(self, idx):
        return self.texts[idx], self.labels[idx]

# Create dataset
dataset = SpamDataset(preprocessed_spam_data['text'], preprocessed_spam_data['label'])

# Split sizes
dataset_len = len(dataset)
train_len = int(dataset_len * 0.7)
val_len = int(dataset_len * 0.2)
test_len = dataset_len - train_len - val_len  # ensure total matches

# Split
train_dataset, val_dataset, test_dataset = random_split(dataset, [train_len, val_len, test_len])
print(f"Train size: {len(train_dataset)}, Val size: {len(val_dataset)}, Test size: {len(test_dataset)}")
print("Tain, Val, Test ratio:" , len(train_dataset)/dataset_len, len(val_dataset)/dataset_len, len(test_dataset)/dataset_len)

Train size: 1082, Val size: 309, Test size: 156
Tain, Val, Test ratio: 0.6994182288299935 0.1997414350355527 0.10084033613445378


---

###  what do we require from lab 04?
1. word2vec model target_embedding layer
2. input to this layer is the index value of each word
3. mapping of word->index is given by word2idx() dictionary
4. word2idx() requires lab 04 dataset and creating vocab fromn it and then creating word2idx() dictionary
5. to handle oov add <|pad|> token to voacb and then create word2idx , then train word2vec algo

In [57]:
import os
import sys
sys.path.append('/home/mudasir/ankit/NLP')
from custom_word2vec import preprocess

dataset_path = "/home/mudasir/ankit/NLP/bbc"

topic = os.listdir(dataset_path)

del topic[topic.index('README.TXT')]

file_paths = []

for t in topic:
    for file_name in os.listdir(os.path.join(dataset_path, t)):
        file_path = os.path.join(dataset_path, t, file_name)
        file_paths.append(file_path)

docs = []
for file_path in file_paths:
    doc = []
    with open(file_path, 'r', encoding="utf-8", errors="ignore") as file:
        for line in file:
            line = preprocess.text_preprocessing(line)
            line = re.sub(r'\n', '', line)
            doc.append(line)
    docs.append(' '.join(doc))

corpus = []
for doc in docs:
    for word in doc.split():
        corpus.append(word)

corpus.append("UNK") # for oov words



In [58]:
from custom_word2vec import Utility
u = Utility(corpus)
vocab, word2idx, idx2word, vocab_size , embedding_dim = u.return_essentials()

In [112]:
# getting pretrained embeddings from the saved word2vec model
import sys
sys.path.append("/home/mudasir/ankit/NLP")
import torch
from custom_word2vec import SkipGramNegSampling  # class must be importable

word2vec_model = SkipGramNegSampling(vocab_size , embedding_dim)
word2vec_model.load_state_dict(torch.load("/home/mudasir/ankit/NLP/custom_kipgram_model.pth"))


<All keys matched successfully>

In [113]:
pretrained_embeddings = word2vec_model.target_embeddings

In [114]:
print(pretrained_embeddings.weight)  # should be (vocab_size, embedding_dim)

Parameter containing:
tensor([[ 0.0422,  0.0661, -0.1444,  ...,  0.0638,  0.0198, -0.1553],
        [ 0.0556,  0.0378, -0.0227,  ..., -0.1640,  0.0979, -0.0769],
        [ 0.0385, -0.1291,  0.1482,  ..., -0.0168, -0.0524, -0.1406],
        ...,
        [ 0.2367, -0.0843, -0.0098,  ..., -0.7584, -0.3040, -0.3808],
        [-0.0635,  0.4360,  0.1280,  ...,  0.0303, -0.2759, -0.4220],
        [ 0.1453,  0.1627, -0.2864,  ..., -0.1431,  0.3130,  0.4248]],
       requires_grad=True)


### RNN model

In [115]:
import torch 
import torch.nn as nn


class RNNModel(nn.Module):
    def __init__(self, hidden_size, output_size, vocab_size=None, embed_size=None,n_layers=1, bidirectional=False, pretrained_embeddings=None):
        super(RNNModel, self).__init__()
        if pretrained_embeddings is not None:
            vocab_size, embed_size = pretrained_embeddings.weight.shape
            self.embedding = nn.Embedding.from_pretrained(pretrained_embeddings.weight, freeze=False)
        else:
            self.embedding = nn.Embedding(vocab_size, embed_size)

        self.rnn = nn.RNN(embed_size, hidden_size, num_layers=n_layers,bidirectional=bidirectional, batch_first=True)
        self.fc = nn.Linear(hidden_size * (2 if bidirectional else 1), output_size)
        self.softmax = nn.Softmax(dim=1)

    def forward(self, x):
        x = self.embedding(x)
        rnn_out, _ = self.rnn(x)
        out = rnn_out[:, -1, :]  # last time step
        out = self.fc(out)
        out = self.softmax(out)
        return out


In [116]:
def custom_collate_fn(batch):
    texts, labels = zip(*batch)
    lengths = [len(text) for text in texts]
    max_length = max(lengths)
    
    padded_texts = []
    for text in texts:
        # encode text
        encoded_text = [word2idx.get(word, word2idx["UNK"]) for word in text]
        encoded_tensor = torch.tensor(encoded_text, dtype=torch.long)
        # pad
        padded_text = torch.cat([encoded_tensor, torch.zeros(max_length - len(encoded_tensor), dtype=torch.long)])
        padded_texts.append(padded_text)
    
    return torch.stack(padded_texts), torch.tensor(labels, dtype=torch.long), torch.tensor(lengths, dtype=torch.long)


In [117]:
# Train, test, val dataloaders
from torch.utils.data import DataLoader
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True, collate_fn=custom_collate_fn)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False, collate_fn=custom_collate_fn)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False, collate_fn=custom_collate_fn)

In [118]:
train_loader.dataset[0]

(['six',
  'chance',
  'win',
  'cash',
  'NUM',
  'NUMNUM',
  'pound',
  'txt',
  'cshNUM',
  'send',
  'NUM',
  'cost',
  'NUMpday',
  'NUMdays',
  'NUM',
  'tsandcs',
  'apply',
  'reply',
  'hl',
  'NUM',
  'info'],
 1)

In [119]:
model = RNNModel(hidden_size=128, output_size=2, pretrained_embeddings=pretrained_embeddings, bidirectional=False)

In [120]:
emb = model.embedding.weight.data
print("NaN in embedding:", torch.isnan(emb).any())


NaN in embedding: tensor(False)


#### Normal Training

In [125]:
def train_rnn_model(model, train_loader, val_loader, num_epochs=10, learning_rate=0.001 , device = None , fold = None):
    criterion = nn.CrossEntropyLoss() 
    if device is None:
        device = torch.device("cuda:4" if torch.cuda.is_available() else "cpu")
    model.to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)

    for epoch in range(num_epochs):
        model.train()
        total_train_loss = 0
        total_val_loss = 0
        for texts, labels , _ in train_loader:
            texts, labels = texts.to(device), labels.to(device)
            optimizer.zero_grad()
            outputs = model(texts)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            total_train_loss += loss.item()

        avg_train_loss = total_train_loss / len(train_loader)

        for texts, labels , _ in val_loader:
            texts, labels = texts.to(device), labels.to(device)
            model.eval()
            with torch.no_grad():
                outputs = model(texts)
                loss = criterion(outputs, labels)
                total_val_loss += loss.item()
        
        avg_val_loss = total_val_loss / len(val_loader)
        
        print(f"Epoch {epoch+1}/{num_epochs}, Train Loss: {avg_train_loss:.4f}, Val Loss: {avg_val_loss:.4f}")
    if fold is not None:
        torch.save(model.state_dict(), f"rnn_model_fold{fold}.pth")
        print(f"Model saved as rnn_model_fold{fold}.pth")
    else:
        torch.save(model.state_dict(), "rnn_model.pth")
        print("Model saved as rnn_model.pth")

In [122]:
device  = torch.device("cuda:4" if torch.cuda.is_available() else "cpu")
train_rnn_model(model, train_loader, val_loader, num_epochs=10, learning_rate=0.001, device=device)

Epoch 1/10, Train Loss: 0.6106, Val Loss: 0.6249
Epoch 2/10, Train Loss: 0.5241, Val Loss: 0.5343
Epoch 3/10, Train Loss: 0.4729, Val Loss: 0.4458
Epoch 4/10, Train Loss: 0.4047, Val Loss: 0.4476
Epoch 5/10, Train Loss: 0.5102, Val Loss: 0.5668
Epoch 6/10, Train Loss: 0.5286, Val Loss: 0.5637
Epoch 7/10, Train Loss: 0.5289, Val Loss: 0.5606
Epoch 8/10, Train Loss: 0.5269, Val Loss: 0.5606
Epoch 9/10, Train Loss: 0.5265, Val Loss: 0.5606
Epoch 10/10, Train Loss: 0.5265, Val Loss: 0.5606
Model saved as rnn_model.pth


In [ ]:
import sklearn
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score , confusion_matrix

def evaluate_rnn_model(model , test_loader , device = None , fold = None):
    if device is None:
        device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model.to(device)
    model.eval()
    all_preds = []
    all_labels = []
    with torch.no_grad():
        for texts, labels , _ in test_loader:
            texts, labels = texts.to(device), labels.to(device)
            outputs = model(texts)
            _, preds = torch.max(outputs, 1)
            all_preds.extend(preds.detach().cpu().numpy())
            all_labels.extend(labels.detach().cpu().numpy())
    
    accuracy = accuracy_score(all_labels, all_preds)
    precision = precision_score(all_labels, all_preds, average='weighted')
    recall = recall_score(all_labels, all_preds, average='weighted')
    f1 = f1_score(all_labels, all_preds, average='weighted')
    cm = confusion_matrix(all_labels, all_preds)
    if fold is not None:
        print(f"Confusion Matrix for {fold+1} fold:")
        print(cm)
        print(f"Test Accuracy for {fold+1} fold: {accuracy:.4f}")
        print(f"Test Precision for {fold+1} fold: {precision:.4f}")
        print(f"Test Recall for {fold+1} fold: {recall:.4f}")
        print(f"Test F1 Score for {fold+1} fold: {f1:.4f}")
    else:
        print("Confusion Matrix for")
        print(cm)
        print(f"Test Accuracy: {accuracy:.4f}")
        print(f"Test Precision: {precision:.4f}")
        print(f"Test Recall: {recall:.4f}")
        print(f"Test F1 Score: {f1:.4f}")
    print()
    return accuracy, precision, recall, f1, cm

In [ ]:
_ , _ , _ , _ , _ = evaluate_rnn_model(model , test_loader , device = device)

Confusion Matrix for
[[57 25]
 [21 53]]
Test Accuracy: 0.7051
Test Precision: 0.7064
Test Recall: 0.7051
Test F1 Score: 0.7053


#### K-fold Cross validation

In [135]:
from sklearn.model_selection import KFold
from torch.utils.data import random_split , DataLoader , Subset

dataset = SpamDataset(preprocessed_spam_data['text'], preprocessed_spam_data['label'])

train_dataset_len = int(0.9 * len(dataset))


train_dataset, test_dataset = random_split(dataset, [train_dataset_len, len(dataset) - train_dataset_len])

kf = KFold(n_splits=5, shuffle=True, random_state=42)

accuracys, precisions, recalls, f1s, cms = [], [], [], [], []

for fold , (train_idx , val_idx) in enumerate(kf.split(train_dataset)):
    print(f"Fold {fold+1}")
    train_subset = Subset(train_dataset, train_idx)
    val_subset = Subset(train_dataset, val_idx)

    train_loader = DataLoader(train_subset, batch_size=32, shuffle=True, collate_fn=custom_collate_fn)
    val_loader = DataLoader(val_subset, batch_size=32, shuffle=False, collate_fn=custom_collate_fn)
    test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False, collate_fn=custom_collate_fn)

    model = RNNModel(hidden_size=128, output_size=2, pretrained_embeddings=pretrained_embeddings, bidirectional=False)

    device  = torch.device("cuda:4" if torch.cuda.is_available() else "cpu")
    train_rnn_model(model, train_loader, val_loader, num_epochs=10, learning_rate=0.001, device=device , fold = fold+1)
    accuracy, precision, recall, f1, cm  = evaluate_rnn_model(model , test_loader , device = device , fold = fold+1 )
    accuracys.append(accuracy)
    precisions.append(precision)
    recalls.append(recall)
    f1s.append(f1)
    cms.append(cm)
    print("--------------------------------------------------")
    print("Avarage accuracy : " , sum(accuracys) / len(accuracys))
    print("Avarage precision : " , sum(precisions) / len(precisions))
    print("Avarage recall : " , sum(recalls) / len(recalls))     
    print("Avarage f1 : " , sum(f1s)/len(f1s)) 
    print()
    print("--------------------------------------------------")


Fold 1
Epoch 1/10, Train Loss: 0.6721, Val Loss: 0.6495
Epoch 2/10, Train Loss: 0.6086, Val Loss: 0.7524
Epoch 3/10, Train Loss: 0.6087, Val Loss: 0.7052
Epoch 4/10, Train Loss: 0.7028, Val Loss: 0.6997
Epoch 5/10, Train Loss: 0.6540, Val Loss: 0.6268
Epoch 6/10, Train Loss: 0.6610, Val Loss: 0.7829
Epoch 7/10, Train Loss: 0.7106, Val Loss: 0.6884
Epoch 8/10, Train Loss: 0.6848, Val Loss: 0.6793
Epoch 9/10, Train Loss: 0.6852, Val Loss: 0.6645
Epoch 10/10, Train Loss: 0.6364, Val Loss: 0.6291
Model saved as rnn_model_fold1.pth
Confusion Matrix for 2 fold:
[[80  4]
 [48 23]]
Test Accuracy for 2 fold: 0.6645
Test Precision for 2 fold: 0.7289
Test Recall for 2 fold: 0.6645
Test F1 Score for 2 fold: 0.6240
--------------------------------------------------
Avarage accuracy :  0.6645161290322581
Avarage precision :  0.7289127837514934
Avarage recall :  0.6645161290322581
Avarage f1 :  0.6240177872731564

--------------------------------------------------
Fold 2
Epoch 1/10, Train Loss: 0.627

/home/mudasir/miniconda3/envs/ankit_NLP/lib/python3.10/site-packages/sklearn/metrics/_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])


Epoch 2/10, Train Loss: 0.5810, Val Loss: 0.6394
Epoch 3/10, Train Loss: 0.5495, Val Loss: 0.5247
Epoch 4/10, Train Loss: 0.4543, Val Loss: 0.4595
Epoch 5/10, Train Loss: 0.4745, Val Loss: 0.5268
Epoch 6/10, Train Loss: 0.5561, Val Loss: 0.6282
Epoch 7/10, Train Loss: 0.5673, Val Loss: 0.5993
Epoch 8/10, Train Loss: 0.5686, Val Loss: 0.5992
Epoch 9/10, Train Loss: 0.5683, Val Loss: 0.5992
Epoch 10/10, Train Loss: 0.5713, Val Loss: 0.6624
Model saved as rnn_model_fold4.pth
Confusion Matrix for 5 fold:
[[33 51]
 [ 0 71]]
Test Accuracy for 5 fold: 0.6710
Test Precision for 5 fold: 0.8085
Test Recall for 5 fold: 0.6710
Test F1 Score for 5 fold: 0.6427
--------------------------------------------------
Avarage accuracy :  0.603225806451613
Avarage precision :  0.5937164016780629
Avarage recall :  0.603225806451613
Avarage f1 :  0.5376017298356275

--------------------------------------------------
Fold 5
Epoch 1/10, Train Loss: 0.6195, Val Loss: 0.5937
Epoch 2/10, Train Loss: 0.5138, Val Lo

---

### CNN Model

In [146]:
import torch.nn as nn
from torch.nn import Conv1d

class CNNModel(nn.Module):
    def __init__(self, num_classes, vocab_size=None, embed_size=None, pretrained_embeddings=None):
        super(CNNModel, self).__init__()
        if pretrained_embeddings is not None:
            vocab_size, embed_size = pretrained_embeddings.weight.shape
            self.embedding = nn.Embedding.from_pretrained(pretrained_embeddings.weight, freeze=False)
        else:
            self.embedding = nn.Embedding(vocab_size, embed_size)

        self.conv1 = Conv1d(in_channels=embed_size, out_channels=200, kernel_size=3, padding=1)
        self.conv2 = Conv1d(in_channels=200, out_channels=100, kernel_size=3, padding=1)
        self.conv3 = Conv1d(in_channels=100, out_channels=10, kernel_size=3, padding=1)
        self.relu = nn.ReLU()
        self.pool = nn.MaxPool1d(kernel_size=2)
        self.fc = nn.Linear(10 * embed_size, num_classes)
        self.softmax = nn.Softmax(dim=1)

    def forward(self, x):
        x = self.embedding(x)  # (batch_size, seq_length, embed_size)
        x = x.permute(0, 2, 1)  # (batch_size, embed_size, seq_length)

        x1 = self.pool(self.relu(self.conv1(x)))  # (batch_size, out_channels, seq_length)
        x2 = self.pool(self.relu(self.conv2(x1)))
        x3 = self.pool(self.relu(self.conv3(x2)))

        x = self.fc(torch.flatten(x3, start_dim=1))
        x = self.softmax(x)
        return x




In [147]:
def train_cnn_model(model, train_loader, val_loader, num_epochs=10, learning_rate=0.001 , device = None , fold = None):
    criterion = nn.CrossEntropyLoss() 
    if device is None:
        device = torch.device("cuda:4" if torch.cuda.is_available() else "cpu")
    model.to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)

    for epoch in range(num_epochs):
        model.train()
        total_train_loss = 0
        total_val_loss = 0
        for texts, labels , _ in train_loader:
            texts, labels = texts.to(device), labels.to(device)
            optimizer.zero_grad()
            outputs = model(texts)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            total_train_loss += loss.item()

        avg_train_loss = total_train_loss / len(train_loader)

        for texts, labels , _ in val_loader:
            texts, labels = texts.to(device), labels.to(device)
            model.eval()
            with torch.no_grad():
                outputs = model(texts)
                loss = criterion(outputs, labels)
                total_val_loss += loss.item()
        
        avg_val_loss = total_val_loss / len(val_loader)
        
        print(f"Epoch {epoch+1}/{num_epochs}, Train Loss: {avg_train_loss:.4f}, Val Loss: {avg_val_loss:.4f}")
    if fold is not None:
        torch.save(model.state_dict(), f"cnn_model_fold{fold}.pth")
        print(f"Model saved as cnn_model_fold{fold}.pth")
    else:
        torch.save(model.state_dict(), "cnn_model.pth")
        print("Model saved as cnn_model.pth")

In [148]:
import sklearn
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score , confusion_matrix

def evaluate_cnn_model(model , test_loader , device = None , fold = None):
    if device is None:
        device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model.to(device)
    model.eval()
    all_preds = []
    all_labels = []
    with torch.no_grad():
        for texts, labels , _ in test_loader:
            texts, labels = texts.to(device), labels.to(device)
            outputs = model(texts)
            _, preds = torch.max(outputs, 1)
            all_preds.extend(preds.detach().cpu().numpy())
            all_labels.extend(labels.detach().cpu().numpy())
    
    accuracy = accuracy_score(all_labels, all_preds)
    precision = precision_score(all_labels, all_preds, average='weighted')
    recall = recall_score(all_labels, all_preds, average='weighted')
    f1 = f1_score(all_labels, all_preds, average='weighted')
    cm = confusion_matrix(all_labels, all_preds)
    if fold is not None:
        print(f"Confusion Matrix for {fold+1} fold:")
        print(cm)
        print(f"Test Accuracy for {fold+1} fold: {accuracy:.4f}")
        print(f"Test Precision for {fold+1} fold: {precision:.4f}")
        print(f"Test Recall for {fold+1} fold: {recall:.4f}")
        print(f"Test F1 Score for {fold+1} fold: {f1:.4f}")
    else:
        print("Confusion Matrix for")
        print(cm)
        print(f"Test Accuracy: {accuracy:.4f}")
        print(f"Test Precision: {precision:.4f}")
        print(f"Test Recall: {recall:.4f}")
        print(f"Test F1 Score: {f1:.4f}")
    print()
    return accuracy, precision, recall, f1, cm

In [149]:
from sklearn.model_selection import KFold
from torch.utils.data import random_split , DataLoader , Subset

dataset = SpamDataset(preprocessed_spam_data['text'], preprocessed_spam_data['label'])

train_dataset_len = int(0.9 * len(dataset))


train_dataset, test_dataset = random_split(dataset, [train_dataset_len, len(dataset) - train_dataset_len])

kf = KFold(n_splits=5, shuffle=True, random_state=42)

accuracys, precisions, recalls, f1s, cms = [], [], [], [], []

for fold , (train_idx , val_idx) in enumerate(kf.split(train_dataset)):
    print(f"Fold {fold+1}")
    train_subset = Subset(train_dataset, train_idx)
    val_subset = Subset(train_dataset, val_idx)

    train_loader = DataLoader(train_subset, batch_size=32, shuffle=True, collate_fn=custom_collate_fn)
    val_loader = DataLoader(val_subset, batch_size=32, shuffle=False, collate_fn=custom_collate_fn)
    test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False, collate_fn=custom_collate_fn)

    model = CNNModel(num_classes=2, vocab_size=vocab_size, embed_size=embedding_dim, pretrained_embeddings=pretrained_embeddings)

    device  = torch.device("cuda:4" if torch.cuda.is_available() else "cpu")
    train_cnn_model(model, train_loader, val_loader, num_epochs=10, learning_rate=0.001, device=device , fold = fold+1)
    accuracy, precision, recall, f1, cm  = evaluate_cnn_model(model , test_loader , device = device , fold = fold+1 )
    accuracys.append(accuracy)
    precisions.append(precision)
    recalls.append(recall)
    f1s.append(f1)
    cms.append(cm)
    print("--------------------------------------------------")
    print("Avarage accuracy : " , sum(accuracys) / len(accuracys))
    print("Avarage precision : " , sum(precisions) / len(precisions))
    print("Avarage recall : " , sum(recalls) / len(recalls))     
    print("Avarage f1 : " , sum(f1s)/len(f1s)) 
    print()
    print("--------------------------------------------------")


Fold 1


RuntimeError: mat1 and mat2 shapes cannot be multiplied (32x20 and 3000x2)